# Lab 03.3 – XADC Periodic Sampling and Data Logging

## Objectives
- Sample XADC measurements periodically using Python.
- Use `time.monotonic()` to schedule measurements.
- Record temperature and supply voltage data.
- Calculate basic statistics.
- Plot the acquired data and save it to a CSV file.

## Hardware
- Digilent Zybo legacy board
- PYNQ 3.0.1
- No external sensor or additional hardware is required.

## Background
A monitoring application normally reads sensors at a defined sampling interval instead of as fast as possible. This experiment combines the XADC measurements from Lab03.1 and Lab03.2 with the timing concepts introduced in Lab02.

`time.monotonic()` is used because it is intended for elapsed-time measurements and is not affected by changes to the system wall clock.


## Exercise 1 – Initialize the XADC access functions


In [ ]:
from pathlib import Path
import re
import time
import csv
import statistics
import matplotlib.pyplot as plt

iio_root = Path("/sys/bus/iio/devices")
xadc_path = None

for dev in sorted(iio_root.glob("iio:device*")):
    name_file = dev / "name"
    if name_file.exists() and name_file.read_text().strip() == "xadc":
        xadc_path = dev
        break

if xadc_path is None:
    raise RuntimeError("XADC IIO device not found.")

def read_temperature_celsius():
    raw = float((xadc_path / "in_temp0_raw").read_text().strip())
    offset = float((xadc_path / "in_temp0_offset").read_text().strip())
    scale = float((xadc_path / "in_temp0_scale").read_text().strip())
    return (raw + offset) * scale / 1000.0

def parameter_file(raw_file, parameter):
    base = raw_file.name[:-4]
    candidates = [xadc_path / f"{base}_{parameter}"]
    m = re.match(r"^(in_voltage\d+)(?:_[A-Za-z0-9]+)?$", base)
    if m:
        candidates.append(xadc_path / f"{m.group(1)}_{parameter}")
    candidates.append(xadc_path / f"in_voltage_{parameter}")
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

def read_voltage(raw_file):
    raw = float(raw_file.read_text().strip())
    scale_file = parameter_file(raw_file, "scale")
    if scale_file is None:
        raise RuntimeError(f"No scale found for {raw_file.name}")
    scale = float(scale_file.read_text().strip())
    offset_file = parameter_file(raw_file, "offset")
    offset = float(offset_file.read_text().strip()) if offset_file else 0.0
    return (raw + offset) * scale / 1000.0

voltage_raw_files = sorted(xadc_path.glob("in_voltage*_raw"))
vccint_file = next((f for f in voltage_raw_files if "vccint" in f.name), None)

if vccint_file is None:
    raise RuntimeError("VCCINT channel not found.")

print("XADC initialized.")
print("Temperature:", f"{read_temperature_celsius():.2f} °C")
print("VCCINT:", f"{read_voltage(vccint_file):.4f} V")


## Exercise 2 – Acquire a time series

Acquire 30 samples at one-second intervals. The next sampling deadline is calculated from the original start time, reducing accumulated drift compared with repeatedly sleeping for exactly one second after each measurement.


In [ ]:
SAMPLES = 30
INTERVAL = 1.0

elapsed_s = []
temperatures_c = []
vccint_v = []

start = time.monotonic()

for i in range(SAMPLES):
    target = start + i * INTERVAL
    delay = target - time.monotonic()
    if delay > 0:
        time.sleep(delay)

    now = time.monotonic()
    temperature = read_temperature_celsius()
    voltage = read_voltage(vccint_file)

    elapsed_s.append(now - start)
    temperatures_c.append(temperature)
    vccint_v.append(voltage)

    print(f"{i + 1:02d}: t={elapsed_s[-1]:6.2f} s  T={temperature:6.2f} °C  VCCINT={voltage:.4f} V")


## Exercise 3 – Calculate measurement statistics


In [ ]:
print("Temperature")
print(f"  mean = {statistics.mean(temperatures_c):.2f} °C")
print(f"  min  = {min(temperatures_c):.2f} °C")
print(f"  max  = {max(temperatures_c):.2f} °C")

print("\nVCCINT")
print(f"  mean = {statistics.mean(vccint_v):.4f} V")
print(f"  min  = {min(vccint_v):.4f} V")
print(f"  max  = {max(vccint_v):.4f} V")

if len(elapsed_s) > 1:
    actual_intervals = [elapsed_s[i] - elapsed_s[i - 1] for i in range(1, len(elapsed_s))]
    print("\nSampling interval")
    print(f"  requested = {INTERVAL:.3f} s")
    print(f"  mean      = {statistics.mean(actual_intervals):.3f} s")
    print(f"  min       = {min(actual_intervals):.3f} s")
    print(f"  max       = {max(actual_intervals):.3f} s")


## Exercise 4 – Plot the measurements


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(elapsed_s, temperatures_c, marker="o")
plt.xlabel("Time (s)")
plt.ylabel("Temperature (°C)")
plt.title("Zynq Die Temperature")
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(elapsed_s, vccint_v, marker="o")
plt.xlabel("Time (s)")
plt.ylabel("VCCINT (V)")
plt.title("Zynq Core Supply Voltage")
plt.grid(True)
plt.show()


## Exercise 5 – Save the data to CSV

The file is created in the same directory as the notebook unless a different path is specified.


In [ ]:
csv_name = "lab03_3_xadc_log.csv"

with open(csv_name, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["time_s", "temperature_C", "vccint_V"])
    for row in zip(elapsed_s, temperatures_c, vccint_v):
        writer.writerow(row)

print("Saved:", csv_name)


## Questions
1. Why is a defined sampling interval useful in a monitoring system?
2. Why is `time.monotonic()` preferable to wall-clock time for scheduling periodic samples?
3. Why does the code calculate each deadline from the original start time instead of simply calling `time.sleep(INTERVAL)` after every measurement?
4. Are the small variations in temperature and VCCINT necessarily evidence of a hardware fault? Explain.
5. What additional information is preserved by saving the samples to CSV instead of only displaying a graph?
